# Revised ML review

This notebook reads public cached evidence only. It does not train, predict or reopen final labels. Scores are not calibrated probabilities. The original training notebook is preserved at tag `legacy-ml-2026-10-25`.

In [1]:
from pathlib import Path
import sys, json
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'review.py').is_file())
sys.path.insert(0, str(ROOT))
from review import cached_results
from tools.verify_handover import verify
print('Public evidence located')

Public evidence located


In [2]:
checks = verify(ROOT)
print('Frozen files unchanged:', checks['frozen_source_and_references_unchanged'])
print('Exact numerical claims:', checks['exact_claims_checked'])

Frozen files unchanged: True
Exact numerical claims: 290


In [3]:
result = cached_results(ROOT)
population = result['population']
print('Cell-days:', population['rows'], '| positive targets:', population['positives'])
for name, row in result['policies'].items():
    print(name, '| recall:', round(row['recall']*100, 2), '| precision:', round(row['precision']*100, 2), '| false alerts:', row['false_positives'], '| misses:', row['false_negatives'])

Cell-days: 7783 | positive targets: 49
recall_80_policy | recall: 32.65 | precision: 29.09 | false alerts: 39 | misses: 33
recall_90_policy | recall: 69.39 | precision: 1.38 | false alerts: 2434 | misses: 15
persistence | recall: 32.65 | precision: 29.09 | false alerts: 39 | misses: 33


In [4]:
contract = json.loads((ROOT / 'docs/technical/model_contract.json').read_text())
print('Ordered inputs:', len(contract['ordered_inputs']))
print('Validated probability:', contract['output_score']['validated_probability'])
for key, value in result['model_ap'].items():
    print(key, 'AP:', round(value, 6))

Ordered inputs: 17
Validated probability: False
training_prevalence AP: 0.006296
persistence AP: 0.099231
logistic_16 AP: 0.046111
logistic_17 AP: 0.211337
xgboost_16 AP: 0.021912
xgboost_17 AP: 0.106152


## Frozen score map

![Cached first-date score map](../report/figures/frozen_score_map.png)

The date is fixed as the first frozen feature day; colours come from saved scores, not a new prediction pass. This is an illustration, not evidence of reliable warnings. See the map provenance JSON and final report.

## Interpretation

The 80%/90% names are development recall targets. Neither transferred to the final population. The logistic comparison ranked better than XGBoost. Both studies use the same 2019-20 source panel, so this is not independent-season replication. Further tuning needs new independent evaluation data. Member understanding, source checks and final report approval remain human review tasks.